# 08 · DataFrames: el siguiente nivel

**Caso:** una tienda tiene las ventas de 3 meses (`pedidos_enero.csv`, `pedidos_febrero.csv`, `pedidos_marzo.csv`), una tabla de `clientes.csv` y otra de `productos.csv`. Vamos a limpiarlas, combinarlas y analizarlas.

**Lo que vas a aprender:**

1. Lidiar con **NaN** (datos faltantes): detectarlos, quitarlos o rellenarlos.
2. **Concatenar** dataframes: apilar meses, pegar columnas.
3. **Cruzar** tablas con `merge` (joins estilo SQL).
4. **Agrupar y resumir** con `groupby` + `agg`.

---

## 0 · Lo que ya vimos

En la sesión anterior aprendiste a:

- Crear/leer DataFrames y explorar con `head`, `info`, `describe`.
- Seleccionar con `loc` / `iloc`.
- Filtrar con condiciones booleanas.

Lo que **no** vimos y vamos a ver hoy: qué hacer cuando faltan datos, cómo unir varias tablas, y cómo resumir por grupos.

---

## Datos del caso

Las tablas viven en el repo público. Las cargamos desde la URL raw de GitHub.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/nodmintuf/notebooks-ml-sesiones/main/data'

URL_PEDIDOS_ENERO   = f'{BASE}/pedidos_enero.csv'
URL_PEDIDOS_FEBRERO = f'{BASE}/pedidos_febrero.csv'
URL_PEDIDOS_MARZO   = f'{BASE}/pedidos_marzo.csv'
URL_CLIENTES        = f'{BASE}/clientes.csv'
URL_PRODUCTOS       = f'{BASE}/productos.csv'

### Cargar las 5 tablas

In [ ]:
ped_ene = pd.read_csv(URL_PEDIDOS_ENERO)
ped_feb = pd.read_csv(URL_PEDIDOS_FEBRERO)
ped_mar = pd.read_csv(URL_PEDIDOS_MARZO)
clientes  = pd.read_csv(URL_CLIENTES)
productos = pd.read_csv(URL_PRODUCTOS)

print('ped_ene  :', ped_ene.shape)
print('ped_feb  :', ped_feb.shape)
print('ped_mar  :', ped_mar.shape)
print('clientes :', clientes.shape)
print('productos:', productos.shape)

### Echamos un vistazo

In [ ]:
ped_ene.head()

In [ ]:
clientes.head()

In [ ]:
productos

---

## 1 · NaN: el enemigo silencioso

**¿Por qué hay NaN?** Tres motivos típicos:

- **Error humano:** se olvidaron de teclear el precio, no se rellenó un campo.
- **Valor desconocido:** no sabemos quién hizo el pedido anónimo.
- **No aplica:** un cliente sin pedidos no tiene `total_gastado`.

NaN es "veneno" para los modelos: la mayoría de algoritmos fallan o lo ignoran en silencio. Tienes que decidir tú qué hacer.

### 1.1 · Detectar: `isna()`

In [ ]:
# ¿Dónde hay NaN en enero?
ped_ene.isna()

In [ ]:
# ¿Cuántos NaN por columna?
ped_ene.isna().sum()

In [ ]:
# ¿Hay alguna fila con NaN?
ped_ene.isna().any(axis=1)

### 1.2 · Quitar: `dropna()`

In [ ]:
# Por defecto, quita CUALQUIER fila que tenga AL MENOS un NaN
ped_ene.dropna()

In [ ]:
# Solo si TODA la fila es NaN (raro, pero existe)
ped_ene.dropna(how='all')

In [ ]:
# Quitar NaN solo de una columna concreta
ped_ene.dropna(subset=['precio_unit'])

### 1.3 · Rellenar: `fillna()`

In [ ]:
# Con un valor constante
ped_ene['precio_unit'].fillna(0)

In [ ]:
# Con la media de la columna
media = ped_ene['precio_unit'].mean()
ped_ene['precio_unit'].fillna(media)

In [ ]:
# Con la mediana (más robusta si hay valores extremos)
mediana = ped_ene['precio_unit'].median()
ped_ene['precio_unit'].fillna(mediana)

### 1.4 · Propagar valores: `ffill` y `bfill`

In [ ]:
# ffill: propaga el valor ANTERIOR hacia delante
# Útil cuando los datos están ordenados en el tiempo y el valor "se mantiene"
ped_ene['precio_unit'].ffill()

In [ ]:
# bfill: propaga el valor SIGUIENTE hacia atrás
ped_ene['precio_unit'].bfill()

### 1.5 · ¿Cuándo usar cada estrategia?

| Situación | Estrategia |
|---|---|
| El NaN significa "0" (p. ej. un descuento que no se aplicó) | `fillna(0)` |
| Es un valor numérico y quieres mantener la media del conjunto | `fillna(df['col'].mean())` |
| Hay outliers que tiran de la media | `fillna(df['col'].median())` |
| Los datos están ordenados y el valor "se arrastra" (ej. precio del día anterior) | `ffill` |
| Tienes pocas filas y no puedes permitirte perderlas | `fillna` (no `dropna`) |
| La fila está demasiado incompleta para ser útil | `dropna` |

### Ejercicios 1 · NaN

1. En `ped_ene`, ¿cuántas filas tienen al menos un NaN? ¿Y en `ped_feb` y `ped_mar`?
2. Crea `ped_ene_limpio` quitando las filas con `precio_unit` NaN. ¿Cuántas filas te quedan?
3. Crea `ped_ene_precio` rellenando los `precio_unit` NaN con la **mediana del producto** correspondiente (pista: `groupby('producto')['precio_unit'].transform('median')`).

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
# 1
for nombre, df in [('enero', ped_ene), ('febrero', ped_feb), ('marzo', ped_mar)]:
    n = df.isna().any(axis=1).sum()
    print(f'{nombre:8s} → {n} filas con al menos un NaN')

# 2
ped_ene_limpio = ped_ene.dropna(subset=['precio_unit'])
print(f'\nQuedan {len(ped_ene_limpio)} filas')

# 3
medianas = ped_ene.groupby('producto')['precio_unit'].transform('median')
ped_ene_precio = ped_ene.copy()
ped_ene_precio['precio_unit'] = ped_ene_precio['precio_unit'].fillna(medianas)
print(f'\nQuedan {ped_ene_precio["precio_unit"].isna().sum()} NaN en precio_unit')

---

## 2 · Concat: apilar y pegar

**Idea:** juntar varios dataframes, **uno encima de otro** (vertical) o **uno al lado del otro** (horizontal).

### 2.1 · Vertical: apilar filas
Por defecto `concat` apila hacia abajo (mismo esquema).

In [ ]:
# Apilar enero y febrero (ambos tienen las mismas columnas)
pd.concat([ped_ene, ped_feb]).head(10)

In [ ]:
# El índice viene de los originales — quedan duplicados
df = pd.concat([ped_ene, ped_feb])
df.index

### 2.2 · `ignore_index=True`
Resetea el índice para que vaya de 0 a N-1 sin repeticiones.

In [ ]:
df = pd.concat([ped_ene, ped_feb], ignore_index=True)
df.index

### 2.3 · Horizontal: pegar columnas
Con `axis=1` pega dataframes **por el índice** (por filas).

In [ ]:
# Truco: añade una columna y concatena horizontalmente con la parte original
demo_horizontal = pd.concat([ped_ene.head(3),
                             ped_ene.head(3).add_prefix('dup_')], axis=1)
demo_horizontal

### 2.4 · Esquemas diferentes

**Cuidado:** si los dataframes no tienen las mismas columnas, `concat` rellena con `NaN` donde falta.

In [ ]:
# Enero y febrero tienen 6 columnas; marzo tiene 7 (añade 'descuento')
print('Columnas enero :', list(ped_ene.columns))
print('Columnas marzo :', list(ped_mar.columns))

# Al apilar los 3, ¿qué pasa?
df = pd.concat([ped_ene, ped_feb, ped_mar], ignore_index=True)
print('\nNaN por columna del dataframe combinado:')
print(df.isna().sum())

In [ ]:
# Verás que 'descuento' tiene NaN en las filas que vienen de enero/febrero
df.tail(35).head(10)

### Ejercicios 2 · Concat

1. Apila los 3 meses en un único dataframe `pedidos` con `ignore_index=True`. ¿Cuántas filas tiene? ¿Cuántas columnas?
2. Mete una columna `mes` en cada dataframe mensual (`'enero'`, `'febrero'`, `'marzo'`) **antes** de concatenarlos. Así sabrás de qué mes viene cada fila.
3. ¿Cuántos pedidos tienen `descuento` no nulo? (Pista: `descuento > 0`.)

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
# 1
pedidos = pd.concat([ped_ene, ped_feb, ped_mar], ignore_index=True)
print('Shape:', pedidos.shape)

# 2
ped_ene2 = ped_ene.assign(mes='enero')
ped_feb2 = ped_feb.assign(mes='febrero')
ped_mar2 = ped_mar.assign(mes='marzo')
pedidos = pd.concat([ped_ene2, ped_feb2, ped_mar2], ignore_index=True)
print('\nColumnas:', list(pedidos.columns))

# 3
print('\nPedidos con descuento > 0:', (pedidos['descuento'] > 0).sum())

---

## 3 · Merge: cruzar tablas (joins estilo SQL)

**Idea:** juntar dos dataframes **por una columna común** (la "llave"). Es lo que en SQL es un `JOIN`.

Por ejemplo: `pedidos` tiene `cliente_id`. `clientes` tiene `cliente_id` + `nombre` + `ciudad`. Con merge, cada pedido se "enriquece" con los datos de su cliente.

### 3.1 · `merge` básico (inner join por defecto)
Sólo se quedan las filas que **existen en ambas tablas**.

In [ ]:
# inner: solo pedidos cuyo cliente_id está en clientes
m = pd.merge(ped_ene, clientes, on='cliente_id', how='inner')
m.head()

In [ ]:
print('ped_ene:', len(ped_ene))
print('tras inner merge:', len(m))
print('(la diferencia son los pedidos con cliente_id NaN o no registrado)')

### 3.2 · Left join: mantener todos los de la izquierda
Aquí 'izquierda' = `ped_ene`. Nos quedamos con **todos** los pedidos, y los que no tienen cliente rellenan con NaN.

In [ ]:
m = pd.merge(ped_ene, clientes, on='cliente_id', how='left')
m.head()

In [ ]:
# Fíjate: los pedidos con cliente_id NaN o 126/127/128 (no están en clientes)
# se quedan con NaN en nombre, ciudad, segmento
m[m['nombre'].isna()]

### 3.3 · Right y Outer join
- **Right:** mantiene todos los de la derecha (todos los clientes), aunque no tengan pedidos.
- **Outer:** mantiene los de ambas tablas.

In [ ]:
# right: TODOS los clientes aparecen, aunque no tengan pedidos
m = pd.merge(ped_ene, clientes, on='cliente_id', how='right')
m[m['pedido_id'].isna()].head()

### 3.4 · Columnas con nombre distinto: `left_on` / `right_on`
Si las columnas llave no se llaman igual en ambas tablas.

In [ ]:
# Ejemplo: en pedidos la columna es 'producto', en productos también,
# pero supongamos que en 'clientes' la llave fuera 'id' en vez de 'cliente_id'
clientes_renombrado = clientes.rename(columns={'cliente_id': 'id'})
m = pd.merge(ped_ene, clientes_renombrado, left_on='cliente_id', right_on='id')
m.head()

### 3.5 · Múltiples llaves y sufijos
Si hay columnas con el mismo nombre en ambas tablas, pandas añade sufijos.

In [ ]:
# Ejemplo: si por error las dos tablas tuvieran una columna 'nombre'
left  = pd.DataFrame({'id': [1, 2], 'nombre': ['A', 'B'], 'valor': [10, 20]})
right = pd.DataFrame({'id': [1, 2], 'nombre': ['X', 'Y'], 'valor': [100, 200]})
pd.merge(left, right, on='id', suffixes=('_izq', '_der'))

### Ejercicios 3 · Merge

1. Haz un **left join** de `ped_ene` con `clientes` por `cliente_id`. ¿Cuántos pedidos quedan con `nombre` NaN?
2. Haz un **left join** de `ped_ene` con `productos` por `producto`. ¿Tienen todos los pedidos un producto en la tabla de productos?
3. Investiga: ¿cuántos pedidos hay de clientes de **Madrid** en enero? (Pista: tras el merge, filtra por `ciudad == 'Madrid'`.)

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
# 1
m = pd.merge(ped_ene, clientes, on='cliente_id', how='left')
print('Pedidos sin nombre de cliente:', m['nombre'].isna().sum())

# 2
m = pd.merge(ped_ene, productos, on='producto', how='left')
print('Pedidos sin producto en catálogo:', m['categoria'].isna().sum())

# 3
m = pd.merge(ped_ene, clientes, on='cliente_id', how='left')
print('Pedidos de Madrid en enero:', (m['ciudad'] == 'Madrid').sum())

---

## 4 · GroupBy: dividir y resumir

**Idea:** 'split-apply-combine'.

1. **Split:** divides las filas en grupos según una columna.
2. **Apply:** aplicas una operación a cada grupo (suma, media, conteo...).
3. **Combine:** juntas los resultados en una tabla resumen.

### 4.0 · Preparamos `pedidos` con la columna `mes`
(Para que las siguientes celdas funcionen aunque no hayas ejecutado la solución del ejercicio 2.)

In [ ]:
# Construimos (o reconstruimos) el dataframe 'pedidos' con la columna 'mes'
ped_ene_m = ped_ene.assign(mes='enero')
ped_feb_m = ped_feb.assign(mes='febrero')
ped_mar_m = ped_mar.assign(mes='marzo')
pedidos = pd.concat([ped_ene_m, ped_feb_m, ped_mar_m], ignore_index=True)

print('Shape:', pedidos.shape)
print('Columnas:', list(pedidos.columns))

### 4.1 · `groupby` + `size`
¿Cuántas filas hay por grupo?

In [ ]:
print('Pedidos por mes:')
print(pedidos.groupby('mes').size())

### 4.2 · `groupby` + `agg` con una métrica

In [ ]:
# Total facturado por mes (cantidad * precio_unit, con NaN → 0 para no romper)
total_mes = pedidos.assign(importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0)) \
                  .groupby('mes')['importe'].sum()
total_mes

### 4.3 · Múltiples métricas a la vez

In [ ]:
resumen = pedidos.assign(importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0)) \
                   .groupby('mes')['importe'] \
                   .agg(['sum', 'mean', 'count'])
resumen

### 4.4 · Varias columnas a la vez

In [ ]:
# Por cada producto, total facturado y nº de pedidos
resumen_prod = (pedidos
    .assign(importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0))
    .groupby('producto')
    .agg(importe_total=('importe', 'sum'),
         n_pedidos=('pedido_id', 'count'),
         unidades=('cantidad', 'sum'))
    .sort_values('importe_total', ascending=False))
resumen_prod

### 4.5 · Agrupar por varias columnas

In [ ]:
# Por mes Y producto
resumen_mes_prod = (pedidos
    .assign(importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0))
    .groupby(['mes', 'producto'])['importe']
    .sum()
    .unstack()      # pone los productos como columnas
    .fillna(0)
    .round(2))
resumen_mes_prod

### Ejercicios 4 · GroupBy

1. ¿Cuántos pedidos hay por **categoría** de producto? (Necesitarás mergear con `productos` primero.)
2. ¿Cuál es el **ticket medio** (importe medio por pedido) por **ciudad** del cliente?
3. ¿Qué **cliente** ha hecho más pedidos en estos 3 meses? Muestra los 5 primeros.

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
# 0. Mergeamos pedidos con productos y clientes
base = (pedidos
    .merge(productos, on='producto', how='left')
    .merge(clientes, on='cliente_id', how='left')
    .assign(importe=pedidos['cantidad'] * pedidos['precio_unit'].fillna(0)))

# 1
print('Pedidos por categoría:')
print(base.groupby('categoria').size())

# 2
print('\nTicket medio por ciudad:')
print(base.groupby('ciudad')['importe'].mean().round(2).sort_values(ascending=False))

# 3
print('\nTop 5 clientes por nº de pedidos:')
print(base.groupby('nombre').size().sort_values(ascending=False).head(5))

---

## 5 · Mini-práctica integradora

Tienes los 3 meses de pedidos y dos tablas de referencia. Demuestra que dominas las 4 herramientas resolviendo este caso de principio a fin.

**Pasos:**

1. **Carga** los 3 CSV de pedidos y concaténalos en un único `pedidos` con `ignore_index=True`.
2. **Limpia** los `precio_unit` NaN usando la **mediana del producto** correspondiente (groupby + transform).
3. **Calcula** la columna `importe = cantidad * precio_unit` (post-limpieza).
4. **Enriquece** `pedidos` haciendo merge con `clientes` (left, por `cliente_id`) y con `productos` (left, por `producto`).
5. **Responde** con `groupby`:
   a. ¿Importe total por **mes y categoría** (pista: `.unstack()`)?
   b. Top 3 **ciudades** por importe total, con su ticket medio.
   c. ¿Qué **segmento** de cliente (premium / standard) factura más?
6. **Bonus:** ¿cuánto se dejó de facturar por los descuentos de marzo? (Pista: `descuento > 0` sobre el `importe`.)

In [ ]:
# Tu código aquí

**Solución orientativa**

In [ ]:
# 1. Carga y concat
ped_ene = pd.read_csv(URL_PEDIDOS_ENERO)
ped_feb = pd.read_csv(URL_PEDIDOS_FEBRERO)
ped_mar = pd.read_csv(URL_PEDIDOS_MARZO)

# 1b. Añadimos la columna 'mes' a cada uno
ped_ene = ped_ene.assign(mes='enero')
ped_feb = ped_feb.assign(mes='febrero')
ped_mar = ped_mar.assign(mes='marzo')

pedidos = pd.concat([ped_ene, ped_feb, ped_mar], ignore_index=True)

# 2. Limpiar precio_unit con la mediana del producto
mediana_prod = pedidos.groupby('producto')['precio_unit'].transform('median')
pedidos['precio_unit'] = pedidos['precio_unit'].fillna(mediana_prod)

# 3. Importe
pedidos['importe'] = pedidos['cantidad'] * pedidos['precio_unit']

# 4. Merge
clientes  = pd.read_csv(URL_CLIENTES)
productos = pd.read_csv(URL_PRODUCTOS)
base = (pedidos
    .merge(clientes,  on='cliente_id', how='left')
    .merge(productos, on='producto',   how='left'))

# 5a. Importe por mes y categoría
print('Importe por mes y categoría:')
print(base.groupby(['mes', 'categoria'])['importe'].sum().unstack().round(2))
print()

# 5b. Top 3 ciudades
print('Top 3 ciudades por importe total:')
print(base.groupby('ciudad')['importe']
          .agg(['sum', 'mean'])
          .sort_values('sum', ascending=False)
          .head(3)
          .round(2))
print()

# 5c. Segmento que más factura
print('Importe por segmento:')
print(base.groupby('segmento')['importe'].sum().round(2))
print()

# 6. Bonus
desc = base.loc[base['descuento'] > 0, 'importe'].sum()
print(f'Descuentos aplicados en marzo: {desc:.2f} €')